# QUWI Metadata and Writer-Disjoint Splits

This notebook examines the QUWI metadata and constructs reproducible writer-disjoint data splits for later experiments.

## Goals

- Confirm the meaning of each page ID
- Verify writer identities and sample counts
- Define train, validation, and test writer sets
- Ensure that no writer appears in more than one split
- Save reproducible split files for future experiments

In [1]:
import csv
import json
import random
import zipfile
from collections import Counter, defaultdict
from io import TextIOWrapper
from pathlib import Path

In [2]:
QUWI_ROOT = Path.home() / "Documents" / "Handwriting" / "QUWI"
TRAIN_ZIP = QUWI_ROOT / "train.zip"
TEST_ZIP = QUWI_ROOT / "test.zip"

print("QUWI root exists:", QUWI_ROOT.exists())
print("Train archive exists:", TRAIN_ZIP.exists())
print("Test archive exists:", TEST_ZIP.exists())

with zipfile.ZipFile(TRAIN_ZIP) as archive:
    train_members = archive.namelist()

with zipfile.ZipFile(TEST_ZIP) as archive:
    test_members = archive.namelist()

print("Train archive contents:", train_members)
print("Test archive contents:", test_members)

QUWI root exists: True
Train archive exists: True
Test archive exists: True
Train archive contents: ['train.csv']
Test archive contents: ['test.csv']


In [3]:
def read_csv_preview(zip_path, csv_name):
    with zipfile.ZipFile(zip_path) as archive:
        with archive.open(csv_name) as raw_file:
            text_file = TextIOWrapper(raw_file, encoding="utf-8-sig", newline="")
            reader = csv.reader(text_file)
            header = next(reader)
            first_row = next(reader)

    return header, first_row

In [4]:
train_header, train_first_row = read_csv_preview(TRAIN_ZIP, "train.csv")
test_header, test_first_row = read_csv_preview(TEST_ZIP, "test.csv")

keywords = ("writer", "script", "language", "lang", "text", "same", "label", "id")

train_metadata_columns = [
    (index, name)
    for index, name in enumerate(train_header)
    if any(keyword in name.lower() for keyword in keywords)
]

test_metadata_columns = [
    (index, name)
    for index, name in enumerate(test_header)
    if any(keyword in name.lower() for keyword in keywords)
]

print("Train columns:", len(train_header))
print("Test columns:", len(test_header))
print("Headers identical:", train_header == test_header)

print("\nFirst 10 column names:")
print(train_header[:10])

print("\nLast 10 column names:")
print(train_header[-10:])

print("\nPossible train metadata columns:")
print(train_metadata_columns)

print("\nPossible test metadata columns:")
print(test_metadata_columns)

print("\nFirst train row metadata values:")
for index, name in train_metadata_columns:
    print(name, "=", train_first_row[index])

print("\nFirst test row metadata values:")
for index, name in test_metadata_columns:
    print(name, "=", test_first_row[index])

Train columns: 7070
Test columns: 7070
Headers identical: True

First 10 column names:
['writer', 'page_id', 'language', 'same_text', 'tortuosityHist10[0]', 'tortuosityHist10[1]', 'tortuosityHist10[2]', 'tortuosityHist10[3]', 'tortuosityHist10[4]', 'tortuosityHist10[5]']

Last 10 column names:
['directions_hist1a2a3a4a5a6a7a8a9a10_220[210]', 'directions_hist1a2a3a4a5a6a7a8a9a10_220[211]', 'directions_hist1a2a3a4a5a6a7a8a9a10_220[212]', 'directions_hist1a2a3a4a5a6a7a8a9a10_220[213]', 'directions_hist1a2a3a4a5a6a7a8a9a10_220[214]', 'directions_hist1a2a3a4a5a6a7a8a9a10_220[215]', 'directions_hist1a2a3a4a5a6a7a8a9a10_220[216]', 'directions_hist1a2a3a4a5a6a7a8a9a10_220[217]', 'directions_hist1a2a3a4a5a6a7a8a9a10_220[218]', 'directions_hist1a2a3a4a5a6a7a8a9a10_220[219]']

Possible train metadata columns:
[(0, 'writer'), (1, 'page_id'), (2, 'language'), (3, 'same_text')]

Possible test metadata columns:
[(0, 'writer'), (1, 'page_id'), (2, 'language'), (3, 'same_text')]

First train row metada

In [5]:
def read_metadata(zip_path, csv_name):
    rows = []

    with zipfile.ZipFile(zip_path) as archive:
        with archive.open(csv_name) as raw_file:
            text_file = TextIOWrapper(raw_file, encoding="utf-8-sig", newline="")
            reader = csv.reader(text_file)
            next(reader)

            for row in reader:
                rows.append(
                    {
                        "writer": int(row[0]),
                        "page_id": int(row[1]),
                        "language": row[2],
                        "same_text": int(row[3]),
                    }
                )

    return rows


train_metadata = read_metadata(TRAIN_ZIP, "train.csv")
test_metadata = read_metadata(TEST_ZIP, "test.csv")

In [6]:
train_writers = sorted({row["writer"] for row in train_metadata})
test_writers = sorted({row["writer"] for row in test_metadata})

train_writer_counts = Counter(row["writer"] for row in train_metadata)
test_writer_counts = Counter(row["writer"] for row in test_metadata)

train_samples_per_writer = Counter(train_writer_counts.values())
test_samples_per_writer = Counter(test_writer_counts.values())

page_mapping = defaultdict(set)

for row in train_metadata + test_metadata:
    page_mapping[row["page_id"]].add(
        (row["language"], row["same_text"])
    )

print("Train rows:", len(train_metadata))
print("Test rows:", len(test_metadata))
print("Total rows:", len(train_metadata) + len(test_metadata))

print("\nTrain writers:", len(train_writers))
print("Train writer range:", train_writers[0], "to", train_writers[-1])

print("\nTest writers:", len(test_writers))
print("Test writer range:", test_writers[0], "to", test_writers[-1])

print("\nWriter overlap:", len(set(train_writers) & set(test_writers)))
print("Train samples per writer:", dict(train_samples_per_writer))
print("Test samples per writer:", dict(test_samples_per_writer))

print("\nPage mapping:")
for page_id in sorted(page_mapping):
    print(page_id, sorted(page_mapping[page_id]))

Train rows: 1128
Test rows: 772
Total rows: 1900

Train writers: 282
Train writer range: 1 to 282

Test writers: 193
Test writer range: 283 to 475

Writer overlap: 0
Train samples per writer: {4: 282}
Test samples per writer: {4: 193}

Page mapping:
1 [('Arabic', 0)]
2 [('Arabic', 1)]
3 [('English', 0)]
4 [('English', 1)]


In [7]:
IMAGE_DIR = QUWI_ROOT / "extracted" / "images"

all_metadata = train_metadata + test_metadata

metadata_filenames = [
    f"{row['writer']:04d}_{row['page_id']}.jpg"
    for row in all_metadata
]

image_filenames = sorted(
    path.name
    for path in IMAGE_DIR.glob("*.jpg")
)

missing_images = sorted(
    set(metadata_filenames) - set(image_filenames)
)

unreferenced_images = sorted(
    set(image_filenames) - set(metadata_filenames)
)

duplicate_metadata_records = (
    len(metadata_filenames) - len(set(metadata_filenames))
)

print("Metadata records:", len(metadata_filenames))
print("Image files:", len(image_filenames))
print("Duplicate metadata records:", duplicate_metadata_records)
print("Missing images:", len(missing_images))
print("Unreferenced images:", len(unreferenced_images))

Metadata records: 1900
Image files: 1900
Duplicate metadata records: 0
Missing images: 0
Unreferenced images: 0


In [8]:
SEED = 42
VALIDATION_FRACTION = 0.20

validation_writer_count = round(
    len(train_writers) * VALIDATION_FRACTION
)

random_generator = random.Random(SEED)

validation_writers = sorted(
    random_generator.sample(
        train_writers,
        validation_writer_count,
    )
)

development_train_writers = sorted(
    set(train_writers) - set(validation_writers)
)

official_test_writers = test_writers

print("Development train writers:", len(development_train_writers))
print("Validation writers:", len(validation_writers))
print("Official test writers:", len(official_test_writers))

print(
    "Train-validation overlap:",
    len(set(development_train_writers) & set(validation_writers)),
)

print(
    "Train-test overlap:",
    len(set(development_train_writers) & set(official_test_writers)),
)

print(
    "Validation-test overlap:",
    len(set(validation_writers) & set(official_test_writers)),
)

print(
    "Total assigned writers:",
    len(
        set(development_train_writers)
        | set(validation_writers)
        | set(official_test_writers)
    ),
)

print("\nValidation writer IDs:")
print(validation_writers)

Development train writers: 226
Validation writers: 56
Official test writers: 193
Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0
Total assigned writers: 475

Validation writer IDs:
[4, 13, 14, 16, 17, 23, 24, 36, 37, 41, 45, 48, 50, 52, 53, 58, 64, 72, 80, 82, 84, 88, 99, 102, 108, 111, 112, 113, 115, 117, 120, 126, 136, 137, 141, 143, 149, 151, 173, 175, 177, 182, 184, 186, 187, 190, 194, 195, 215, 217, 230, 233, 236, 259, 275, 280]


In [9]:
writer_split_lookup = {
    **{writer: "development_train" for writer in development_train_writers},
    **{writer: "validation" for writer in validation_writers},
    **{writer: "official_test" for writer in official_test_writers},
}

split_records = []

for source_split, metadata_rows in [
    ("official_train", train_metadata),
    ("official_test", test_metadata),
]:
    for row in metadata_rows:
        split_records.append(
            {
                "filename": f"{row['writer']:04d}_{row['page_id']}.jpg",
                "writer": row["writer"],
                "page_id": row["page_id"],
                "language": row["language"],
                "same_text": row["same_text"],
                "source_split": source_split,
                "experiment_split": writer_split_lookup[row["writer"]],
            }
        )

split_sample_counts = Counter(
    row["experiment_split"]
    for row in split_records
)

split_writer_counts = {
    split_name: len(
        {
            row["writer"]
            for row in split_records
            if row["experiment_split"] == split_name
        }
    )
    for split_name in [
        "development_train",
        "validation",
        "official_test",
    ]
}

print("Sample counts:", dict(split_sample_counts))
print("Writer counts:", split_writer_counts)

Sample counts: {'development_train': 904, 'validation': 224, 'official_test': 772}
Writer counts: {'development_train': 226, 'validation': 56, 'official_test': 193}


In [10]:
for split_name in [
    "development_train",
    "validation",
    "official_test",
]:
    rows = [
        row
        for row in split_records
        if row["experiment_split"] == split_name
    ]

    language_counts = Counter(
        row["language"]
        for row in rows
    )

    page_counts_by_split = Counter(
        row["page_id"]
        for row in rows
    )

    same_text_counts = Counter(
        row["same_text"]
        for row in rows
    )

    print(f"\n{split_name}")
    print("Languages:", dict(language_counts))
    print("Page IDs:", dict(sorted(page_counts_by_split.items())))
    print("Same-text labels:", dict(sorted(same_text_counts.items())))


development_train
Languages: {'Arabic': 452, 'English': 452}
Page IDs: {1: 226, 2: 226, 3: 226, 4: 226}
Same-text labels: {0: 452, 1: 452}

validation
Languages: {'Arabic': 112, 'English': 112}
Page IDs: {1: 56, 2: 56, 3: 56, 4: 56}
Same-text labels: {0: 112, 1: 112}

official_test
Languages: {'Arabic': 386, 'English': 386}
Page IDs: {1: 193, 2: 193, 3: 193, 4: 193}
Same-text labels: {0: 386, 1: 386}


In [11]:
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "pyproject.toml").exists()

SPLIT_DIR = PROJECT_ROOT / "splits"
SPLIT_DIR.mkdir(exist_ok=True)

split_csv_path = SPLIT_DIR / "quwi_writer_disjoint_split_seed42.csv"

fieldnames = [
    "filename",
    "writer",
    "page_id",
    "language",
    "same_text",
    "source_split",
    "experiment_split",
]

with split_csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(split_records)

print("Split CSV:", split_csv_path)
print("Rows saved:", len(split_records))

Split CSV: /home/arijit/Documents/handwriting-cross-script-research/splits/quwi_writer_disjoint_split_seed42.csv
Rows saved: 1900


In [12]:
split_json_path = SPLIT_DIR / "quwi_writer_disjoint_split_seed42.json"

split_definition = {
    "dataset": "QUWI",
    "seed": SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "official_train_writer_range": [train_writers[0], train_writers[-1]],
    "official_test_writer_range": [test_writers[0], test_writers[-1]],
    "development_train_writers": development_train_writers,
    "validation_writers": validation_writers,
    "official_test_writers": official_test_writers,
    "writer_counts": split_writer_counts,
    "sample_counts": dict(split_sample_counts),
}

with split_json_path.open("w", encoding="utf-8") as file:
    json.dump(split_definition, file, indent=2)

print("Split JSON:", split_json_path)
print("Development train writers:", len(development_train_writers))
print("Validation writers:", len(validation_writers))
print("Official test writers:", len(official_test_writers))

Split JSON: /home/arijit/Documents/handwriting-cross-script-research/splits/quwi_writer_disjoint_split_seed42.json
Development train writers: 226
Validation writers: 56
Official test writers: 193


In [13]:
with split_csv_path.open("r", encoding="utf-8") as file:
    saved_split_rows = list(csv.DictReader(file))

with split_json_path.open("r", encoding="utf-8") as file:
    saved_split_definition = json.load(file)

print("Saved CSV rows:", len(saved_split_rows))
print("CSV columns:", list(saved_split_rows[0].keys()))
print("Saved seed:", saved_split_definition["seed"])
print("Writer counts:", saved_split_definition["writer_counts"])
print("Sample counts:", saved_split_definition["sample_counts"])

assert len(saved_split_rows) == 1900
assert saved_split_definition["seed"] == 42

Saved CSV rows: 1900
CSV columns: ['filename', 'writer', 'page_id', 'language', 'same_text', 'source_split', 'experiment_split']
Saved seed: 42
Writer counts: {'development_train': 226, 'validation': 56, 'official_test': 193}
Sample counts: {'development_train': 904, 'validation': 224, 'official_test': 772}


In [14]:
saved_writers_by_split = defaultdict(set)
saved_samples_per_writer = Counter()

for row in saved_split_rows:
    writer_id = int(row["writer"])
    split_name = row["experiment_split"]

    saved_writers_by_split[split_name].add(writer_id)
    saved_samples_per_writer[writer_id] += 1

saved_train_writers = saved_writers_by_split["development_train"]
saved_validation_writers = saved_writers_by_split["validation"]
saved_test_writers = saved_writers_by_split["official_test"]

print("Saved unique writers:", len(set(saved_samples_per_writer)))
print("Samples per writer:", dict(Counter(saved_samples_per_writer.values())))
print("Train-validation overlap:", len(saved_train_writers & saved_validation_writers))
print("Train-test overlap:", len(saved_train_writers & saved_test_writers))
print("Validation-test overlap:", len(saved_validation_writers & saved_test_writers))

assert len(saved_samples_per_writer) == 475
assert set(saved_samples_per_writer.values()) == {4}
assert not saved_train_writers & saved_validation_writers
assert not saved_train_writers & saved_test_writers
assert not saved_validation_writers & saved_test_writers

Saved unique writers: 475
Samples per writer: {4: 475}
Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0


## Metadata interpretation

Each QUWI writer has four handwriting images:

| Page ID | Language | Text condition |
|---|---|---|
| 1 | Arabic | Variable text |
| 2 | Arabic | Same text |
| 3 | English | Variable text |
| 4 | English | Same text |

The `same_text` value is `0` for variable-text pages and `1` for fixed-text pages. This structure allows later experiments to separate script effects from text-content effects.

## Writer-disjoint split design

The official QUWI split contains writers 1–282 for training and writers 283–475 for testing. The official test writers are kept completely untouched.

For model development, 20% of the official training writers were selected as a validation set using random seed 42.

| Experiment split | Writers | Images |
|---|---:|---:|
| Development train | 226 | 904 |
| Validation | 56 | 224 |
| Official test | 193 | 772 |
| Total | 475 | 1900 |

No writer appears in more than one split. This prevents writer-identity leakage between training, validation, and testing.

The current files define image-level membership only. Genuine and impostor verification pairs will be constructed separately according to each experimental protocol.